In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader,Subset
import torchvision
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
import numpy as np

In [ ]:
import zipfile

drive_loc=Path("/content/drive/MyDrive")
zip_path=drive_loc/"Synthesized_refined.zip"
extract_to=Path("/content/Gesture")

if not extract_to.is_dir():

  extract_to.mkdir(parents=True , exist_ok=True)
else:
  print("It exists")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_to)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_dir_path=Path("Gesture/Synthesized_refined/Refined_gesture_cropped")


In [ ]:
objective_list=["no_gesture","ok","one","palm","peace","three","thumb_index"]

def class_formation_helper_function(objective_list):
  dictionery_class={class_name:index for index,class_name in enumerate(objective_list) }
  return dictionery_class

In [ ]:
print(class_formation_helper_function(objective_list=objective_list))

{'no_gesture': 0, 'ok': 1, 'one': 2, 'palm': 3, 'peace': 4, 'three': 5, 'thumb_index': 6}


In [ ]:
class My_augmentation:
  def __init__(self,image):
    self.image=image

  def train_transform(self):
    img=transforms.functional.resize(self.image,(512,512),transforms.InterpolationMode.BILINEAR)
    if torch.randn(1)<0.5:

      img=transforms.functional.hflip(img)

    if torch.randn(1) <0.5:
      factor_brightness=random.uniform(0.8,1.2)
      img=transforms.functional.adjust_brightness(img,factor_brightness)
    if torch.randn(1)<0.5:
      factor_contrast=random.uniform(0.8,1.2)
      img=transforms.functional.adjust_contrast(img,factor_contrast)

    if torch.randn(1)<0.5:
      rotating_angle=random.uniform(-10,10)
      img=transforms.functional.rotate(img,rotating_angle)

    if torch.randn(1) <0.5:
      angle,shift,scale,shear=transforms.RandomAffine.get_params(
          degrees=(0,0),
          translate=(0.15,0.15),
          scale_ranges=(0.85,1.15),
          shears=None,
          img_size=(512,512)
      )
      img=transforms.functional.affine(
          img,
          angle,
          shift,
          scale,
          shear
      )
    img=transforms.functional.to_tensor(img)

    return img

  def test_transform(self):
    img=transforms.functional.resize(self.image,(512,512),transforms.InterpolationMode.BILINEAR)
    img=transforms.functional.to_tensor(img)
    return img




In [ ]:
class custom_dataset(Dataset):
  def __init__(self,dir_path,objective_list,transform=None):
    super().__init__()
    self.image_paths=sorted(list(dir_path.glob("*/*.jpg")))
    self.transform_type=transform
    self.class_to_idx=class_formation_helper_function(objective_list=objective_list)

  def _imageopener(self,index):
    return Image.open(self.image_paths[index])

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self,index):
    img=self._imageopener(index)
    class_idx=self.class_to_idx[self.image_paths[index].parent.stem]
    my_transforms=My_augmentation(img)

    if self.transform_type=="train":

      return my_transforms.train_transform(),class_idx

    else:
      return my_transforms.test_transform(),class_idx










In [ ]:
dataset_train=custom_dataset(dir_path=file_dir_path,objective_list=objective_list,transform="train")
dataset_test=custom_dataset(dir_path=file_dir_path,objective_list=objective_list,transform=None)

In [ ]:
indices=[i for i in range(0,len(dataset_train))]

In [ ]:
train_idx,test_idx=train_test_split(indices,test_size=0.2,random_state=42)

In [ ]:
train_dataset=Subset(dataset=dataset_train,indices=train_idx)
test_dataset=Subset(dataset=dataset_test,indices=test_idx)


In [ ]:
train_dataset[5][0].unique()

tensor([0.0000, 0.0431, 0.0471, 0.0510, 0.0549, 0.0588, 0.0627, 0.0667, 0.0706,
        0.0745, 0.0784, 0.0824, 0.0863, 0.0902, 0.0941, 0.0980, 0.1020, 0.1059,
        0.1098, 0.1137, 0.1176, 0.1216, 0.1255, 0.1294, 0.1333, 0.1373, 0.1412,
        0.1451, 0.1490, 0.1529, 0.1569, 0.1608, 0.1647, 0.1686, 0.1725, 0.1765,
        0.1804, 0.1843, 0.1882, 0.1922, 0.1961, 0.2000, 0.2039, 0.2078, 0.2118,
        0.2157, 0.2196, 0.2235, 0.2275, 0.2314, 0.2353, 0.2392, 0.2431, 0.2471,
        0.2510, 0.2549, 0.2588, 0.2627, 0.2667, 0.2706, 0.2745, 0.2784, 0.2824,
        0.2863, 0.2902, 0.2941, 0.2980, 0.3020, 0.3059, 0.3098, 0.3137, 0.3176,
        0.3216, 0.3255, 0.3294, 0.3333, 0.3373, 0.3412, 0.3451, 0.3490, 0.3529,
        0.3569, 0.3608, 0.3647, 0.3686, 0.3725, 0.3765, 0.3804, 0.3843, 0.3882,
        0.3922, 0.3961, 0.4000, 0.4039, 0.4078, 0.4118, 0.4157, 0.4196, 0.4235,
        0.4275, 0.4314, 0.4353, 0.4392, 0.4431, 0.4471, 0.4510, 0.4549, 0.4588,
        0.4627, 0.4667, 0.4706, 0.4745, 

In [ ]:
train_dataloader=DataLoader(dataset=train_dataset,batch_size=6,shuffle=True)
test_dataloader=DataLoader(dataset=test_dataset,batch_size=6,shuffle=True)

In [ ]:
class model_1(nn.Module):
  def __init__(self,in_features,hidden_conv_units,out_features,hidden_linear_units):
    super().__init__()
    self.conv_layer_1=nn.Sequential(
        nn.Conv2d(in_channels=in_features,out_channels=hidden_conv_units,kernel_size=(5,5),padding=2,stride=1),
        nn.BatchNorm2d(hidden_conv_units),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_conv_units,out_channels=48,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(48),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=2)
    )
    self.conv_layer_2=nn.Sequential(
        nn.Conv2d(in_channels=48,out_channels=48,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(48),
        nn.ReLU(),
        nn.Conv2d(in_channels=48,out_channels=64,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=2)
    )
    self.conv_layer_3=nn.Sequential(
        nn.Conv2d(in_channels=64,out_channels=64,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(in_channels=64,out_channels=80,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(80),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=2)
    )

    self.conv_layer_4=nn.Sequential(
        nn.Conv2d(in_channels=80,out_channels=80,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(80),
        nn.ReLU(),
        nn.Conv2d(in_channels=80,out_channels=96,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(96),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=2)
    )
    self.conv_layer_5=nn.Sequential(
        nn.Conv2d(in_channels=96,out_channels=108,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(108),
        nn.ReLU(),
        nn.Conv2d(in_channels=108,out_channels=128,kernel_size=(5,5),padding=1,stride=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=2),
        nn.MaxPool2d(kernel_size=(2,2),stride=2)
    )
    self.linear_layer=nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=4608,out_features=hidden_linear_units),
        nn.ReLU(),
        nn.Linear(in_features=hidden_linear_units,out_features=out_features)
    )
  def forward(self,X_tensor):
    return self.linear_layer(self.conv_layer_5(self.conv_layer_4(self.conv_layer_3(self.conv_layer_2(self.conv_layer_1(X_tensor))))))

In [ ]:

in_features=3
hidden_conv_features=32
hidden_linear_features=38
out_features=7


In [ ]:
model=model_1(in_features=in_features,hidden_conv_units=hidden_conv_features,hidden_linear_units=hidden_linear_features,out_features=out_features)
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

model_1(
  (conv_layer_1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 48, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer_2): Sequential(
    (0): Conv2d(48, 48, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(48, 64, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer_3): Sequenti

In [ ]:
def accuracy_fn(y_pred,y_true):
  count=torch.eq(y_pred,y_true).sum()
  percentage=(count/len(y_pred))*100

  return percentage.item()

In [ ]:
def train_step(model,dataloader,loss_fn,optimizer,accuracy_fn,device):
  model.train()

  avg_loss=0
  avg_accuracy=0

  for X_train,Y_train in tqdm(dataloader):
    X_train,Y_train=X_train.to(device),Y_train.to(device)
    y_raw_logits=model(X_train)
    loss=loss_fn(y_raw_logits,Y_train)
    avg_loss+=loss.item()
    y_refined_logits=torch.softmax(y_raw_logits,dim=1).argmax(dim=1)
    avg_accuracy+=accuracy_fn(y_pred=y_refined_logits,y_true=Y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  avg_loss/=len(dataloader)
  avg_accuracy/=len(dataloader)

  return avg_loss,avg_accuracy



In [ ]:
def test_step(model,dataloader,loss_fn,accuracy_fn,device):
  model.eval()

  avg_loss=0
  avg_accuracy=0

  with torch.inference_mode():
    for X_test,Y_test in tqdm(dataloader):
      X_test,Y_test=X_test.to(device),Y_test.to(device)
      y_raw_logits=model(X_test)
      loss=loss_fn(y_raw_logits,Y_test)
      avg_loss+=loss.item()
      y_refined_logits=torch.softmax(y_raw_logits,dim=1).argmax(dim=1)
      avg_accuracy+=accuracy_fn(y_pred=y_refined_logits,y_true=Y_test)




  avg_loss/=len(dataloader)
  avg_accuracy/=len(dataloader)

  return avg_loss,avg_accuracy



In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model.parameters(),lr=0.00003)


In [ ]:
epochs=1

for epoch in tqdm(range(epochs)):
  avg_train_loss,avg_train_accuracy=train_step(model=model,dataloader=train_dataloader,loss_fn=loss_fn,accuracy_fn=accuracy_fn,optimizer=optimizer,device=device)
  avg_test_loss,avg_test_accuracy=test_step(model=model,dataloader=test_dataloader,loss_fn=loss_fn,accuracy_fn=accuracy_fn,device=device)
  print("Avg_train_loss:",avg_train_loss,"Avg_test_loss:",avg_test_loss)
  print("Avg_train_accuracy",avg_train_accuracy,"Avg_test_accuracy:",avg_test_accuracy)




  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1322 [00:00<?, ?it/s]

  0%|          | 0/331 [00:00<?, ?it/s]

Avg_train_loss: 0.438654642169957 Avg_test_loss: 0.24689753848378437
Avg_train_accuracy 84.79576588109835 Avg_test_accuracy: 91.84290140849225


In [ ]:
epochs=1

for epoch in tqdm(range(epochs)):
  avg_train_loss,avg_train_accuracy=train_step(model=model,dataloader=train_dataloader,loss_fn=loss_fn,accuracy_fn=accuracy_fn,optimizer=optimizer,device=device)
  avg_test_loss,avg_test_accuracy=test_step(model=model,dataloader=test_dataloader,loss_fn=loss_fn,accuracy_fn=accuracy_fn,device=device)
  print("Avg_train_loss:",avg_train_loss,"Avg_test_loss:",avg_test_loss)
  print("Avg_train_accuracy",avg_train_accuracy,"Avg_test_accuracy:",avg_test_accuracy)




  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1322 [00:00<?, ?it/s]

  0%|          | 0/331 [00:00<?, ?it/s]

Avg_train_loss: 0.27886517207527817 Avg_test_loss: 0.14483974318707987
Avg_train_accuracy 91.01109557505275 Avg_test_accuracy: 95.11581139694168


In [ ]:
epochs=3

for epoch in tqdm(range(epochs)):
  avg_train_loss,avg_train_accuracy=train_step(model=model,dataloader=train_dataloader,loss_fn=loss_fn,accuracy_fn=accuracy_fn,optimizer=optimizer,device=device)
  avg_test_loss,avg_test_accuracy=test_step(model=model,dataloader=test_dataloader,loss_fn=loss_fn,accuracy_fn=accuracy_fn,device=device)
  print("Avg_train_loss:",avg_train_loss,"Avg_test_loss:",avg_test_loss)
  print("Avg_train_accuracy",avg_train_accuracy,"Avg_test_accuracy:",avg_test_accuracy)




  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/1322 [00:00<?, ?it/s]

  0%|          | 0/331 [00:00<?, ?it/s]

Avg_train_loss: 0.24559413047532766 Avg_test_loss: 0.12429816965252904
Avg_train_accuracy 91.70448934581023 Avg_test_accuracy: 96.07250812910834


  0%|          | 0/1322 [00:00<?, ?it/s]

  0%|          | 0/331 [00:00<?, ?it/s]

Avg_train_loss: 0.22691489209739052 Avg_test_loss: 0.11602381569807477
Avg_train_accuracy 92.72566923178992 Avg_test_accuracy: 96.57603274803508


  0%|          | 0/1322 [00:00<?, ?it/s]

  0%|          | 0/331 [00:00<?, ?it/s]

Avg_train_loss: 0.2101450668083949 Avg_test_loss: 0.10678179298203176
Avg_train_accuracy 92.93999093383235 Avg_test_accuracy: 96.92849996514911


In [ ]:
torch.save(model.state_dict(),"model_synthesized_94_97.pth")